# Guía de EDA con R: Nacimientos en Chile 2020–2022

**Dataset:** Serie de nacimientos registrados en Chile entre 2020 y 2022 — 561.554 registros con 25 variables demográficas. Fuente: DEIS (Departamento de Estadísticas e Información de Salud), MINSAL.

**Ideas de la guía**
- Estadística descriptiva: medidas de tendencia central, dispersión y posición
- Exploratory Data Analysis (EDA): visualizar, explorar y preguntar sobre datos reales
- R con tidyverse: `dplyr`, `ggplot2`, `lubridate`, `janitor`

Cada sección plantea una conjetura antes del código. Piensa un poco sobre cómo imaginas el resultado antes de ejecutar.

---

## Cómo usar este notebook

1. En Google Colab, debes ir a **Entorno de ejecución → Cambiar tipo de entorno de ejecución → R**
2. Sube el archivo `Serie_Nacimientos_2020_2022.csv` usando el panel de archivos (ícono de carpeta en el panel izquierdo → botón de subir archivo)
3. Ejecuta las celdas en orden con **Shift+Enter** o el botón ▶
4. Lee la conjetura de cada sección *antes* de ejecutar el código

---

## Configuración inicial

Ejecuta las siguientes celdas antes de comenzar. Solo es necesario una vez por sesión.

In [ ]:
# Instalar paquetes (puede tomar 1-2 minutos la primera vez)
install.packages(c("tidyverse", "janitor", "lubridate"), quiet = TRUE)

In [ ]:
suppressPackageStartupMessages({
  library(tidyverse)
  library(janitor)
  library(lubridate)
})

# Tamaño y tema por defecto para todos los gráficos
options(repr.plot.width = 10, repr.plot.height = 5)
theme_set(theme_minimal(base_size = 13))

### Cargar el dataset

El archivo CSV usa punto y coma (`;`) como separador y codificación ISO-8859-1 (Latin-1). Hay que indicárselo explícitamente a R, porque asume la coma  como separador y UTF-8.

In [ ]:
nacimientos_raw <- read_delim(
  "Serie_Nacimientos_2020_2022.csv",
  delim = ";",
  locale = locale(encoding = "latin1"),
  show_col_types = FALSE
)

cat("Dataset cargado:",
    format(nrow(nacimientos_raw), big.mark = ".", decimal.mark = ","),
    "filas x",
    ncol(nacimientos_raw), "columnas\n")

---

## Parte 0 — R en 10 minutos

Si ya sabes Python u otro lenguaje, esta parte te pone al día con R en pocas celdas. Si ya conoces R, puedes saltar directamente a la Parte 1.

In [ ]:
# Tipos de datos básicos
numero <- 3.14          # numeric  (equivale a float en Python)
entero <- 42L           # integer  (el sufijo L indica entero)
texto  <- "hola"        # character (equivale a str)
logico <- TRUE          # logical   (equivale a bool)

cat(class(numero), class(entero), class(texto), class(logico), "\n")

# Conversión explícita
as.numeric("3.14")   # character → numeric
as.character(42)     # numeric → character
as.logical(0)        # 0 → FALSE, cualquier otro número → TRUE

In [ ]:
# El vector es la estructura atómica de R
# No existe el "escalar": un número solo es un vector de longitud 1
edades <- c(22, 25, 30, 28, 35)

# Operaciones vectorizadas (se aplican a todos los elementos en paralelo)
edades + 1             # suma 1 a cada elemento
edades > 25            # devuelve vector lógico
edades[edades > 25]    # filtrar por condición booleana
edades[c(1, 3)]        # filtrar por índice (empieza en 1, no en 0)

cat("Media:", mean(edades),
    "\nMediana:", median(edades),
    "\nDE:", round(sd(edades), 2), "\n")

In [ ]:
# tibble: tabla con columnas de distintos tipos (versión moderna del data.frame)
personas <- tibble(
  nombre = c("Ana", "Luis", "María", "Pedro"),
  edad   = c(22, 35, 28, 31),
  activo = c(TRUE, FALSE, TRUE, TRUE)
)
personas

# Acceso a columnas
personas$edad           # como vector
personas[["nombre"]]    # equivalente

In [ ]:
# El pipe %>% encadena transformaciones
# x %>% f(y)  es equivalente a  f(x, y)
#
# Sin pipe:
round(mean(sqrt(abs(c(-4, 9, -16, 25)))), 2)

# Con pipe (se lee de izquierda a derecha):
c(-4, 9, -16, 25) %>% abs() %>% sqrt() %>% mean() %>% round(2)

In [ ]:
# Los 5 verbos clave de dplyr
# filter()    → filtrar filas         (≈ WHERE en SQL)
# select()    → seleccionar columnas  (≈ SELECT en SQL)
# mutate()    → crear/modificar columnas
# summarise() → calcular un resumen (colapsa filas)
# group_by()  → agrupar antes de summarise (≈ GROUP BY en SQL)

personas %>%
  filter(activo == TRUE) %>%                        # solo activos
  mutate(decada = (edad %/% 10) * 10) %>%           # nueva columna
  select(nombre, edad, decada) %>%                  # solo estas columnas
  arrange(desc(edad))                               # ordenar por edad

---

## Parte 1 — Conociendo los datos

Antes de calcular, hay que entender qué hay en el dataset: cuántos registros, qué variables, qué tipos, cuántos valores faltantes.

### Conjetura

> **Antes de ejecutar:** ¿Cuántos nacimientos anuales estimarías que hay en Chile? ¿Esperarías algún efecto del COVID-19 en los números de 2020 y 2021? Escribe tu predicción antes de continuar.

In [ ]:
# Dimensiones y nombres de columnas
cat("Filas:", format(nrow(nacimientos_raw), big.mark = ".", decimal.mark = ","), "\n")
cat("Columnas:", ncol(nacimientos_raw), "\n\n")
names(nacimientos_raw)

In [ ]:
# Vista compacta: nombre, tipo y primeros valores de cada columna
glimpse(nacimientos_raw)

In [ ]:
# Primeras filas
head(nacimientos_raw, 5)

In [ ]:
# Conteo de nacimientos por año
nacimientos_raw %>%
  count(ANO_NAC, name = "nacimientos") %>%
  mutate(
    porcentaje  = round(100 * nacimientos / sum(nacimientos), 1),
    nacimientos = format(nacimientos, big.mark = ".", decimal.mark = ",")
  )

**Interpretación:** Los tres años tienen recuentos similares (~185.000–190.000 nacimientos/año). Chile venía con una tendencia a la baja en la natalidad desde 2014 aproximadamente. El efecto COVID en los nacimientos se ve mejor en el análisis temporal (Parte 5), donde se mira mes a mes.

### Diccionario de variables

La mayoría de las columnas usan **códigos numéricos**. Acá está el diccionario resumido:

| Variable | Descripción | Codificación |
|---|---|---|
| `MES_NAC`, `ANO_NAC` | Mes y año de nacimiento | 1–12, 2020–2022 |
| `SEXO` | Sexo del recién nacido | 1=Masc, 2=Fem |
| `TIPO_PARTO` | Tipo de parto | 1=Único, 2=Gemelar, 3=Triple o más |
| `TIPO_ATEN` | Profesional que atendió | 1=Médico, 2=Matrona, 3=Otro calificado, 4=Sin profesional |
| `PARTO_LOCAL` | Lugar del parto | 1=Estab. salud, 2=Domicilio, 3=En tránsito |
| `SEMANAS` | Semanas de gestación | Numérico (18–44) |
| `RANGO_PESO` | Peso al nacer en gramos | Texto: `<1500`, `1500 - 2499`, `2500 - 2999`, `3000 - 3999`, `4000 y más` |
| `TALLA` | Talla al nacer en cm | Numérico (30–60) |
| `NIVEL_MADRE` | Nivel educacional de la madre | 1=Básica, 2=Media, 3=Técnica sup., 4=Universitaria, 5=Postgrado |
| `EST_CIV_MADRE` | Estado civil de la madre | 1=Soltera, 2=Casada, 4=Separada, 6=Unión civil |
| `ACTIV_MADRE` | Actividad laboral de la madre | 0=No trabaja, 1=Trabaja |
| `GLOSA_REGION_RESIDENCIA` | Nombre de la región de residencia | Texto |
| `9` en variables numéricas | Código para "No especificado" | → se convierte a `NA` |

In [ ]:
# Resumen estadístico automático de todas las columnas
# Para numéricas: mín, Q1, mediana, media, Q3, máx, NAs
summary(nacimientos_raw)

In [ ]:
# Mapa de valores faltantes (NA verdaderos en R)
# Nota: los códigos 9, "Ignorado", etc. todavía no están convertidos a NA
nacimientos_raw %>%
  summarise(across(everything(), ~sum(is.na(.)))) %>%
  pivot_longer(everything(), names_to = "variable", values_to = "n_NA") %>%
  filter(n_NA > 0) %>%
  arrange(desc(n_NA))

In [ ]:
# Explorar los valores únicos de variables categóricas
# Esto es lo primero que debes hacer antes de interpretar los códigos
for (col in c("SEXO", "TIPO_PARTO", "TIPO_ATEN", "NIVEL_MADRE", "EST_CIV_MADRE")) {
  cat("\n=====", col, "====\n")
  print(nacimientos_raw %>% count(!!sym(col), sort = TRUE))
}

In [ ]:
# RANGO_PESO: la única variable con valores de texto descriptivo
nacimientos_raw %>%
  count(RANGO_PESO, sort = TRUE)

In [ ]:
# Crear dataset limpio con variables decodificadas
# A partir de aquí usaremos 'nacimientos' en lugar de 'nacimientos_raw'

nacimientos <- nacimientos_raw %>%
  mutate(
    # Sexo
    sexo = case_when(
      SEXO == 1 ~ "Masculino",
      SEXO == 2 ~ "Femenino",
      TRUE ~ NA_character_
    ),
    # Tipo de parto (número de bebés en el parto)
    tipo_parto = case_when(
      TIPO_PARTO == 1 ~ "Único",
      TIPO_PARTO == 2 ~ "Gemelar",
      TIPO_PARTO == 3 ~ "Triple o más",
      TRUE ~ NA_character_
    ),
    # Profesional que atendió el parto
    tipo_atencion = case_when(
      TIPO_ATEN == 1 ~ "Médico",
      TIPO_ATEN == 2 ~ "Matrona",
      TIPO_ATEN == 3 ~ "Otro calificado",
      TIPO_ATEN == 4 ~ "Sin profesional",
      TRUE ~ NA_character_
    ),
    # Nivel educacional de la madre
    nivel_educ_madre = case_when(
      NIVEL_MADRE == 1 ~ "Básica",
      NIVEL_MADRE == 2 ~ "Media",
      NIVEL_MADRE == 3 ~ "Técnica superior",
      NIVEL_MADRE == 4 ~ "Universitaria",
      NIVEL_MADRE == 5 ~ "Postgrado",
      TRUE ~ NA_character_
    ),
    nivel_educ_madre = fct_relevel(nivel_educ_madre,
      "Básica", "Media", "Técnica superior", "Universitaria", "Postgrado"
    ),
    # Estado civil de la madre
    est_civil_madre = case_when(
      EST_CIV_MADRE == 1 ~ "Soltera",
      EST_CIV_MADRE == 2 ~ "Casada",
      EST_CIV_MADRE == 4 ~ "Separada/Div.",
      EST_CIV_MADRE == 6 ~ "Unión civil",
      TRUE ~ NA_character_
    ),
    # Variables numéricas: convertir a entero y filtrar valores inválidos
    semanas  = if_else(SEMANAS >= 18 & SEMANAS <= 44,  as.integer(SEMANAS), NA_integer_),
    talla_cm = if_else(TALLA   >= 30 & TALLA   <= 60,  as.integer(TALLA),   NA_integer_),
    # Peso al nacer como factor ordenado (para que los gráficos respeten el orden)
    peso_rango = factor(
      if_else(RANGO_PESO == "Ignorado", NA_character_, RANGO_PESO),
      levels  = c("<1500", "1500 - 2499", "2500 - 2999", "3000 - 3999", "4000 y más"),
      ordered = TRUE
    ),
    # Fecha para el análisis temporal
    fecha = make_date(ANO_NAC, MES_NAC, 1L)
  )

cat("Dataset limpio listo:",
    format(nrow(nacimientos), big.mark = ".", decimal.mark = ","), "filas\n")
glimpse(nacimientos %>% select(sexo, tipo_parto, tipo_atencion,
                                nivel_educ_madre, semanas, talla_cm, peso_rango, fecha))

---

## Parte 2 — Estadística Descriptiva

La estadística descriptiva resume los datos en números concisos. Las medidas más importantes son:
- **Tendencia central:** ¿dónde está el "centro" de los datos? (media, mediana, moda)
- **Dispersión:** ¿cuánto varían los datos alrededor del centro? (varianza, DE, IQR)
- **Posición:** ¿qué valor separa el X% inferior del resto? (percentiles, cuantiles)

### Conjetura

> **Antes de ejecutar:** ¿Cuántas semanas de gestación dura un embarazo típico? ¿Esperarías que la media y la mediana sean iguales, o una distribución asimétrica? ¿Y la talla de un recién nacido típico?

In [ ]:
# R no tiene función para la moda estadística — la definimos
moda <- function(x, na.rm = TRUE) {
  if (na.rm) x <- x[!is.na(x)]
  tabla <- table(x)
  as.numeric(names(tabla)[which.max(tabla)])
}

# Medidas de tendencia central para SEMANAS
cat("=== Semanas de gestación ===\n")
cat("Media:   ", round(mean(nacimientos$semanas,   na.rm = TRUE), 2), "\n")
cat("Mediana: ", median(nacimientos$semanas, na.rm = TRUE), "\n")
cat("Moda:    ", moda(nacimientos$semanas), "\n")

In [ ]:
# Medidas de dispersión
s <- nacimientos$semanas

cat("=== Dispersión de semanas ===\n")
cat("Varianza:         ", round(var(s, na.rm = TRUE), 4), "\n")
cat("Desviación estándar:", round(sd(s, na.rm = TRUE), 4), "\n")
cat("Rango (max - min):", diff(range(s, na.rm = TRUE)), "\n")
cat("IQR (Q3 - Q1):    ", IQR(s, na.rm = TRUE), "\n")

# La DE y el IQR miden 'dispersión típica' pero de maneras distintas:
# - DE: sensible a outliers (usa la media)
# - IQR: robusto a outliers (usa los cuartiles)

In [ ]:
# Cuantiles y percentiles
# quantile(x, probs = p) devuelve el valor que deja al p% de los datos por debajo
cat("=== Percentiles de semanas ===\n")
quantile(nacimientos$semanas, probs = c(0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95), na.rm = TRUE)

In [ ]:
# Tabla de resumen completa con summarise()
# summarise() colapsa todas las filas en una sola con los cálculos que pidas
nacimientos %>%
  summarise(
    n_total        = n(),
    n_validos_sem  = sum(!is.na(semanas)),
    media_semanas  = round(mean(semanas,  na.rm = TRUE), 2),
    mediana_semanas = median(semanas, na.rm = TRUE),
    de_semanas     = round(sd(semanas,    na.rm = TRUE), 2),
    iqr_semanas    = IQR(semanas,         na.rm = TRUE),
    media_talla    = round(mean(talla_cm, na.rm = TRUE), 2),
    mediana_talla  = median(talla_cm,     na.rm = TRUE),
    de_talla       = round(sd(talla_cm,   na.rm = TRUE), 2)
  )

In [ ]:
# group_by() + summarise(): estadísticas por grupo
# ¿Hay diferencias entre nacimientos masculinos y femeninos?
nacimientos %>%
  filter(!is.na(sexo)) %>%
  group_by(sexo) %>%
  summarise(
    n               = n(),
    media_semanas   = round(mean(semanas,  na.rm = TRUE), 2),
    mediana_semanas = median(semanas,      na.rm = TRUE),
    de_semanas      = round(sd(semanas,    na.rm = TRUE), 2),
    media_talla     = round(mean(talla_cm, na.rm = TRUE), 2),
    de_talla        = round(sd(talla_cm,   na.rm = TRUE), 2),
    .groups = "drop"
  )

**Interpretación:**
- La media y la mediana de semanas son muy similares (~38–39), lo que sugiere una distribución aproximadamente simétrica.
- El IQR es pequeño (≈ 2 semanas), indicando que la gran mayoría de los partos ocurre en un rango estrecho.
- Las diferencias entre sexos son pequeñas pero consistentes: los varones tienden a ser ligeramente más grandes al nacer.

---

## Parte 3 — EDA Univariada

El análisis univariado estudia **una variable a la vez**: su distribución, su forma, sus valores extremos. Es el punto de partida de cualquier EDA.

### Conjetura: variables numéricas

> **Antes de ejecutar:** ¿Cómo imaginas el histograma de semanas de gestación? ¿Simétrico? ¿Sesgado? ¿Con una cola larga a la izquierda (partos prematuros)? ¿Y la talla al nacer?

In [ ]:
# Histograma de semanas de gestación
nacimientos %>%
  filter(!is.na(semanas)) %>%
  ggplot(aes(x = semanas)) +
  geom_histogram(binwidth = 1, fill = "#1976D2", color = "white", alpha = 0.85) +
  geom_vline(xintercept = 37, linetype = "dashed", color = "#E53935", linewidth = 0.8) +
  annotate("text", x = 35.8, y = Inf, label = "Prematuro\n(< 37)",
           vjust = 1.4, hjust = 1.0, color = "#E53935", size = 3.5) +
  labs(title = "Distribución de semanas de gestación",
       subtitle = paste0(format(sum(!is.na(nacimientos$semanas)), big.mark = ".", decimal.mark = ","), " registros válidos"),
       x = "Semanas de gestación", y = "Frecuencia")

In [ ]:
# Curva de densidad + boxplot (para ver la forma y los outliers juntos)
# Con 561k filas, geom_jitter sería muy lento — usamos violin + boxplot
set.seed(42)
muestra_sem <- nacimientos %>% filter(!is.na(semanas)) %>% sample_n(10000)

p_densidad <- ggplot(muestra_sem, aes(x = semanas)) +
  geom_density(fill = "#BBDEFB", color = "#1565C0", alpha = 0.7) +
  labs(title = "Densidad de semanas de gestación",
       subtitle = "Muestra de 10.000 registros",
       x = "Semanas", y = "Densidad")

p_boxplot <- ggplot(muestra_sem, aes(y = semanas, x = "")) +
  geom_violin(fill = "#E3F2FD", color = "#1565C0") +
  geom_boxplot(width = 0.15, fill = "#1976D2", alpha = 0.8,
               outlier.size = 0.5, outlier.alpha = 0.3) +
  labs(title = "Boxplot de semanas", x = NULL, y = "Semanas")

# Mostrar los dos gráficos en una fila
cowplot_available <- require("cowplot", quietly = TRUE)
if (cowplot_available) {
  cowplot::plot_grid(p_densidad, p_boxplot, ncol = 2)
} else {
  print(p_densidad)
  print(p_boxplot)
}

In [ ]:
# Histograma de talla al nacer
nacimientos %>%
  filter(!is.na(talla_cm)) %>%
  ggplot(aes(x = talla_cm)) +
  geom_histogram(binwidth = 1, fill = "#388E3C", color = "white", alpha = 0.85) +
  labs(title = "Distribución de talla al nacer",
       x = "Talla (cm)", y = "Frecuencia")

**Observación:** La distribución de semanas tiene una cola izquierda pronunciada (partos muy prematuros son posibles pero raros), y un pico marcado en 38–39 semanas. La distribución de talla es más simétrica.

Esto ilustra una diferencia entre **media** y **mediana**: cuando hay una cola larga, la media se desplaza en esa dirección.

### Conjetura: variables categóricas

> **Antes de ejecutar:** ¿Qué proporción de nacimientos esperas que sean niñas vs. niños? ¿Cuál es el rango de peso más frecuente?

In [ ]:
# Distribución por sexo
# tabyl() de janitor: tabla de frecuencias limpia con porcentajes
nacimientos %>%
  filter(!is.na(sexo)) %>%
  tabyl(sexo) %>%
  adorn_pct_formatting(digits = 1)

In [ ]:
nacimientos %>%
  filter(!is.na(sexo)) %>%
  count(sexo) %>%
  mutate(pct = round(100 * n / sum(n), 1)) %>%
  ggplot(aes(x = sexo, y = n, fill = sexo)) +
  geom_col(show.legend = FALSE, alpha = 0.9, width = 0.5) +
  geom_text(aes(label = paste0(pct, "%")), vjust = -0.5, size = 4.5) +
  scale_fill_manual(values = c("Femenino" = "#D81B60", "Masculino" = "#1565C0")) +
  scale_y_continuous(labels = scales::label_number(big.mark = ".", decimal.mark = ",")) +
  labs(title = "Nacimientos por sexo", x = NULL, y = "Nacimientos")

In [ ]:
# Distribución por tipo de parto
nacimientos %>%
  filter(!is.na(tipo_parto)) %>%
  tabyl(tipo_parto) %>%
  adorn_pct_formatting(digits = 2)

In [ ]:
# Distribución por rango de peso
# Como es un factor ordenado, los niveles respetan el orden real
nacimientos %>%
  filter(!is.na(peso_rango)) %>%
  count(peso_rango) %>%
  mutate(pct = round(100 * n / sum(n), 1)) %>%
  ggplot(aes(x = peso_rango, y = n)) +
  geom_col(fill = "#7B1FA2", alpha = 0.85) +
  geom_text(aes(label = paste0(pct, "%")), vjust = -0.5, size = 3.5) +
  scale_y_continuous(labels = scales::label_number(big.mark = ".", decimal.mark = ",")) +
  labs(title = "Distribución por rango de peso al nacer",
       x = "Rango de peso (g)", y = "Nacimientos") +
  theme(axis.text.x = element_text(angle = 15, hjust = 1))

---

## Parte 4 — EDA Bivariada

El análisis bivariado estudia la **relación entre dos variables**. Nos permite responder preguntas como: ¿el nivel educacional de la madre se asocia con la duración del embarazo?

### Conjetura

> **Antes de ejecutar:** ¿Esperarías que las madres con mayor nivel educacional tengan embarazos de más o menos semanas? ¿Qué relación esperarías entre semanas de gestación y talla al nacer? ¿Y entre tipo de parto (único vs. gemelar) y el rango de peso?

In [ ]:
# Boxplot: semanas de gestación según nivel educacional de la madre
# Boxplot agrupado: ideal para comparar una variable numérica entre categorías
nacimientos %>%
  filter(!is.na(nivel_educ_madre), !is.na(semanas)) %>%
  ggplot(aes(x = nivel_educ_madre, y = semanas, fill = nivel_educ_madre)) +
  geom_boxplot(show.legend = FALSE, alpha = 0.8, outlier.size = 0.5, outlier.alpha = 0.2) +
  geom_hline(yintercept = 37, linetype = "dashed", color = "#E53935", alpha = 0.7) +
  annotate("text", x = 0.5, y = 36.7, label = "Prematuro", hjust = 0,
           color = "#E53935", size = 3) +
  labs(title = "Semanas de gestación según nivel educacional de la madre",
       x = "Nivel educacional", y = "Semanas de gestación") +
  theme(axis.text.x = element_text(angle = 15, hjust = 1))

In [ ]:
# Tabla de estadísticas condicionadas: nivel educacional → semanas
nacimientos %>%
  filter(!is.na(nivel_educ_madre), !is.na(semanas)) %>%
  group_by(nivel_educ_madre) %>%
  summarise(
    n                = n(),
    media_semanas    = round(mean(semanas), 2),
    mediana_semanas  = median(semanas),
    de_semanas       = round(sd(semanas), 2),
    pct_prematuros   = round(100 * mean(semanas < 37), 1),
    .groups = "drop"
  )

In [ ]:
# Scatterplot: semanas vs. talla al nacer (muestra para rapidez)
set.seed(42)
muestra_biv <- nacimientos %>%
  filter(!is.na(semanas), !is.na(talla_cm)) %>%
  sample_n(8000)

ggplot(muestra_biv, aes(x = semanas, y = talla_cm)) +
  geom_point(alpha = 0.15, color = "#455A64", size = 0.7) +
  geom_smooth(method = "loess", color = "#E53935", se = TRUE, linewidth = 1.2) +
  labs(title = "Relación entre semanas de gestación y talla al nacer",
       subtitle = "Muestra de 8.000 registros + curva LOESS",
       x = "Semanas de gestación", y = "Talla al nacer (cm)")

In [ ]:
# Barras apiladas: ¿cómo varía el rango de peso según el tipo de parto?
# Categórica × Categórica
nacimientos %>%
  filter(!is.na(tipo_parto), !is.na(peso_rango),
         tipo_parto %in% c("Único", "Gemelar")) %>%
  count(tipo_parto, peso_rango) %>%
  group_by(tipo_parto) %>%
  mutate(pct = n / sum(n)) %>%
  ggplot(aes(x = tipo_parto, y = pct, fill = peso_rango)) +
  geom_col(position = "stack") +
  scale_y_continuous(labels = scales::percent) +
  scale_fill_brewer(palette = "RdYlGn", direction = 1) +
  labs(title = "Distribución de peso al nacer por tipo de parto",
       x = "Tipo de parto", y = "Proporción", fill = "Rango de peso")

**Reflexión:** El scatterplot confirma la relación positiva entre semanas y talla (a más semanas, más talla), con una zona de saturación al acercarse a las 40 semanas. Los partos gemelares tienen una distribución de peso más cargada hacia los rangos bajos — ¿por qué podría ser eso?

---

## Parte 5 — EDA Temporal

Los datos tienen dimensión temporal: tres años de nacimientos mes a mes. Esto permite detectar tendencias, estacionalidad y eventos externos.

### Conjetura

> **Antes de ejecutar:** El confinamiento por COVID en Chile comenzó en marzo de 2020. Los bebés concebidos durante los meses de confinamiento nacerían aproximadamente 9 meses después. ¿Esperarías ver un alza o una baja de nacimientos en ese período? ¿Cuándo exactamente en el tiempo?

In [ ]:
# Construir serie mensual de nacimientos
serie_mensual <- nacimientos %>%
  count(fecha, name = "n_nacimientos") %>%
  arrange(fecha)

# Vista previa
print(serie_mensual, n = 5)
cat("\nTotal de meses:", nrow(serie_mensual))

In [ ]:
# Serie temporal mensual con anotaciones de hitos COVID
ggplot(serie_mensual, aes(x = fecha, y = n_nacimientos)) +
  geom_line(color = "#1565C0", linewidth = 0.9) +
  geom_point(color = "#1565C0", size = 2.5) +
  # Confinamiento: marzo 2020 = concepción → diciembre 2020 = nacimiento
  geom_vline(xintercept = as.numeric(make_date(2020, 3, 1)),
             linetype = "dashed", color = "#B71C1C", alpha = 0.8) +
  annotate("text", x = make_date(2020, 4, 15), y = Inf,
           label = "Cuarentena\nChile", vjust = 1.5, color = "#B71C1C", size = 3.2) +
  annotate("segment",
           x    = make_date(2020, 3, 1),  xend = make_date(2020, 12, 1),
           y    = min(serie_mensual$n_nacimientos) * 0.97,
           yend = min(serie_mensual$n_nacimientos) * 0.97,
           color = "#F57F17", linewidth = 1,
           arrow = arrow(ends = "both", type = "open", length = unit(0.08, "inches"))) +
  annotate("text",
           x     = make_date(2020, 7, 15),
           y     = min(serie_mensual$n_nacimientos) * 0.97,
           label = "≈ 9 meses", vjust = 1.5, color = "#F57F17", size = 3) +
  scale_x_date(date_labels = "%b %Y", date_breaks = "3 months") +
  scale_y_continuous(labels = scales::label_number(big.mark = ".", decimal.mark = ",")) +
  labs(title = "Nacimientos mensuales en Chile 2020–2022",
       x = NULL, y = "Nacimientos") +
  theme(axis.text.x = element_text(angle = 45, hjust = 1))

In [ ]:
# Comparar los tres años superpuestos en el mismo eje de mes
meses_label <- c("Ene", "Feb", "Mar", "Abr", "May", "Jun",
                 "Jul", "Ago", "Sep", "Oct", "Nov", "Dic")

nacimientos %>%
  count(ANO_NAC, MES_NAC) %>%
  ggplot(aes(x = MES_NAC, y = n, color = factor(ANO_NAC), group = ANO_NAC)) +
  geom_line(linewidth = 1) +
  geom_point(size = 2.5) +
  scale_x_continuous(breaks = 1:12, labels = meses_label) +
  scale_y_continuous(labels = scales::label_number(big.mark = ".", decimal.mark = ",")) +
  scale_color_manual(values = c("2020" = "#1565C0", "2021" = "#E53935", "2022" = "#2E7D32")) +
  labs(title = "Nacimientos por mes: comparación 2020 vs. 2021 vs. 2022",
       x = "Mes", y = "Nacimientos", color = "Año")

**Interpretación:** La gráfica superpuesta permite ver si el patrón estacional (típicamente con picos en primavera) se repite cada año, y si hay un año que se desvía del patrón habitual.

> **Pregunta para reflexionar:** ¿Ves el efecto COVID en la serie? ¿En qué meses es más notorio? ¿Es un alza o una baja? ¿Coincide con tu conjetura inicial?

---

## Parte 6 — EDA Geográfica

Chile tiene 16 regiones con realidades demográficas y de salud distintas. Analizar datos por región permite detectar inequidades territoriales.

### Conjetura

> **Antes de ejecutar:** ¿Qué región concentraría más nacimientos? ¿Esperarías diferencias en las semanas de gestación entre la Región Metropolitana y las regiones del extremo sur o norte del país?

In [ ]:
# Total de nacimientos por región (ordenado de mayor a menor)
# fct_reorder() ordena los niveles de un factor según otra variable
nacimientos %>%
  count(GLOSA_REGION_RESIDENCIA) %>%
  mutate(region = fct_reorder(GLOSA_REGION_RESIDENCIA, n)) %>%
  ggplot(aes(x = region, y = n)) +
  geom_col(fill = "#00695C", alpha = 0.85) +
  geom_text(aes(label = format(n, big.mark = ".", decimal.mark = ",")), hjust = -0.1, size = 3) +
  coord_flip() +
  scale_y_continuous(labels = scales::label_number(big.mark = ".", decimal.mark = ","),
                     expand = expansion(mult = c(0, 0.12))) +
  labs(title = "Total de nacimientos por región (2020–2022)",
       x = NULL, y = "Nacimientos")

In [ ]:
# Mediana de semanas de gestación por región (gráfico de puntos / lollipop)
nacimientos %>%
  filter(!is.na(semanas)) %>%
  group_by(GLOSA_REGION_RESIDENCIA) %>%
  summarise(
    mediana_semanas  = median(semanas),
    pct_prematuros   = round(100 * mean(semanas < 37), 1),
    n = n(),
    .groups = "drop"
  ) %>%
  mutate(region = fct_reorder(GLOSA_REGION_RESIDENCIA, mediana_semanas)) %>%
  ggplot(aes(x = region, y = pct_prematuros)) +
  geom_segment(aes(xend = region,
                   y    = min(pct_prematuros) - 0.1,
                   yend = pct_prematuros),
               color = "#B0BEC5") +
  geom_point(aes(size = n), color = "#E53935", alpha = 0.8) +
  coord_flip() +
  labs(title = "Tasa de prematurez (< 37 semanas) por región",
       subtitle = "Tamaño del punto proporcional al número de nacimientos",
       x = NULL, y = "% nacimientos prematuros",
       size = "N nacimientos") +
  guides(size = guide_legend(override.aes = list(color = "#E53935")))

---

## Parte 7 — EDA Multivariada

El análisis multivariado combina tres o más variables simultáneamente. Las herramientas típicas son: gráficos con facetas, heatmaps y tablas de resumen agrupadas.

### Conjetura

> **Antes de ejecutar:** ¿Qué combinación de factores estimas que tiene la mayor tasa de prematurez? ¿Cómo esperarías que varíe el número de nacimientos a lo largo del año para cada uno de los tres años?

In [ ]:
# Heatmap: nacimientos por mes y año
# geom_tile() es el geom para heatmaps en ggplot2
meses_label <- c("Ene", "Feb", "Mar", "Abr", "May", "Jun",
                 "Jul", "Ago", "Sep", "Oct", "Nov", "Dic")

nacimientos %>%
  count(ANO_NAC, MES_NAC) %>%
  ggplot(aes(x = factor(MES_NAC), y = factor(ANO_NAC), fill = n)) +
  geom_tile(color = "white", linewidth = 0.5) +
  geom_text(aes(label = format(n, big.mark = ".", decimal.mark = ",")), size = 3.5) +
  scale_fill_gradient(low = "#E3F2FD", high = "#0D47A1") +
  scale_x_discrete(labels = meses_label) +
  labs(title = "Mapa de calor: nacimientos por mes y año",
       x = "Mes", y = "Año", fill = "Nacimientos")

In [ ]:
# Histogramas con facetas: semanas según tipo de parto y sexo
# facet_wrap() genera un panel por cada valor de la variable
nacimientos %>%
  filter(!is.na(sexo), !is.na(semanas),
         tipo_parto %in% c("Único", "Gemelar")) %>%
  ggplot(aes(x = semanas, fill = sexo)) +
  geom_histogram(binwidth = 1, position = "identity", alpha = 0.55) +
  facet_wrap(~tipo_parto, scales = "free_y",
             labeller = labeller(tipo_parto = c(
               "Único" = "Parto único", "Gemelar" = "Parto gemelar"
             ))) +
  scale_fill_manual(values = c("Femenino" = "#D81B60", "Masculino" = "#1565C0")) +
  geom_vline(xintercept = 37, linetype = "dashed", color = "#E53935") +
  labs(title = "Distribución de semanas de gestación por tipo de parto y sexo",
       x = "Semanas", y = "Frecuencia", fill = "Sexo")

In [ ]:
# Tasa de prematurez según nivel educacional de la madre
nacimientos %>%
  filter(!is.na(nivel_educ_madre), !is.na(semanas)) %>%
  group_by(nivel_educ_madre) %>%
  summarise(
    n              = n(),
    pct_prematuros = round(100 * mean(semanas < 37), 1),
    .groups = "drop"
  ) %>%
  ggplot(aes(x = nivel_educ_madre, y = pct_prematuros)) +
  geom_col(fill = "#E53935", alpha = 0.85) +
  geom_text(aes(label = paste0(pct_prematuros, "%")), vjust = -0.5, size = 4) +
  labs(title = "Tasa de prematurez (< 37 sem.) según nivel educacional de la madre",
       x = "Nivel educacional", y = "% nacimientos prematuros") +
  theme(axis.text.x = element_text(angle = 15, hjust = 1))

In [ ]:
# Tabla de resumen multivariable
# Combina: nivel educacional × estadísticas de semanas y peso
nacimientos %>%
  filter(!is.na(nivel_educ_madre), !is.na(semanas)) %>%
  group_by(nivel_educ_madre) %>%
  summarise(
    n                = format(n(), big.mark = ".", decimal.mark = ","),
    media_semanas    = round(mean(semanas), 2),
    de_semanas       = round(sd(semanas), 2),
    pct_prematuros   = round(100 * mean(semanas < 37), 1),
    media_talla      = round(mean(talla_cm, na.rm = TRUE), 1),
    pct_bajo_peso    = round(100 * mean(peso_rango <= "2500 - 2999", na.rm = TRUE), 1),
    .groups = "drop"
  ) %>%
  rename(
    "Nivel educ."   = nivel_educ_madre,
    "N"             = n,
    "Media sem."    = media_semanas,
    "DE sem."       = de_semanas,
    "% prematuros"  = pct_prematuros,
    "Media talla"   = media_talla,
    "% bajo peso"   = pct_bajo_peso
  )

---

## Cierre — Preguntas abiertas para exploración

El EDA no termina con las preguntas que alguien (un profe, por ejemplo) te hizo, si no que termina cuando tú decides que ya entendiste lo suficiente para avanzar. Acá van seis preguntas sin respuesta para que continúes por tu cuenta:

1. **Profesional que atiende por región:** ¿En qué regiones es más frecuente que los partos sean atendidos por matronas en vez de médicos? ¿Coincide con las regiones de mayor o menor urbanización?

2. **Evolución de la tasa de partos gemelares:** ¿Cambió la proporción de partos gemelares entre 2020 y 2022? ¿Qué podría explicar una variación?

3. **Madres jóvenes:** El dataset incluye `GRUPO_ETARIO_MADRE`. ¿Qué distribución de semanas de gestación tienen las madres adolescentes (<20 años) en comparación con las adultas?

4. **Estacionalidad del peso:** ¿El rango de peso al nacer varía según el mes del año? ¿Hay meses con más nacimientos de bajo peso (<2500 g)?

5. **Brecha educacional por región:** ¿Las madres de la Región Metropolitana tienen mayor nivel educacional que las del promedio del país? ¿Cómo se correlaciona el nivel educacional promedio por región con la tasa de prematurez?

6. **Datos faltantes:** La columna `ACTIV_MADRE` tiene un código 9 (no especificado). ¿En qué regiones o grupos de edad es más frecuente el dato faltante? ¿Podría ser que los datos faltantes no sean aleatorios?

---

## Resumen de herramientas aprendidas

| Tarea | Función(es) |
|---|---|
| Ver estructura del dataset | `glimpse()`, `dim()`, `names()`, `head()` |
| Estadísticas descriptivas | `mean()`, `median()`, `sd()`, `IQR()`, `quantile()`, `summary()` |
| Frecuencias categóricas | `count()`, `tabyl()` + `adorn_pct_formatting()` |
| Estadísticas por grupo | `group_by()` + `summarise()` |
| Crear variables | `mutate()` + `case_when()` / `if_else()` |
| Histograma | `geom_histogram()` |
| Densidad | `geom_density()` |
| Boxplot | `geom_boxplot()` |
| Violin plot | `geom_violin()` |
| Barras | `geom_col()` |
| Scatterplot | `geom_point()` + `geom_smooth()` |
| Serie temporal | `geom_line()` + `geom_point()` |
| Heatmap | `geom_tile()` |
| Facetas | `facet_wrap()` |
| Ordenar factores | `fct_reorder()`, `fct_relevel()` |